Awesome 😎 — let’s upgrade your design into a **dynamic multi-agent architecture** where the **Parent Agent** reasons and decides *which specialized MCP agent* (like Weather, Math, or Search) to delegate the task to.

We’ll structure it cleanly, scalable for future new MCPs — you can drop a new agent file into the `mcp_agents/` folder and it will auto-register dynamically.

---

## ⚙️ Project Structure

```
langgraph_project/
│
├── main.py
├── parent_agent.py
│
├── mcp_agents/
│   ├── __init__.py
│   ├── weather_agent.py
│   ├── math_agent.py
│   └── search_agent.py
│
├── .env
└── requirements.txt
```

---

## 🧠 `parent_agent.py`

### → Dynamic MCP Routing (Core Controller)

```python
# parent_agent.py

"""
Parent Agent — routes reasoning dynamically to the appropriate MCP Agent
based on user intent (Weather, Math, Search, etc.)
"""

import importlib
import pkgutil
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool
from dotenv import load_dotenv
import os

# -------------------------------
# Load environment variables
# -------------------------------
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("⚠️ Please set OPENAI_API_KEY in your .env file.")

# -------------------------------
# Initialize base LLM
# -------------------------------
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5, api_key=OPENAI_API_KEY)

# -------------------------------
# Dynamically load MCP agent modules
# -------------------------------
def load_mcp_agents():
    """Auto-discovers all MCP agents in the mcp_agents/ folder."""
    mcp_agents = {}
    package = "mcp_agents"
    for _, name, _ in pkgutil.iter_modules([package]):
        module = importlib.import_module(f"{package}.{name}")
        if hasattr(module, "get_tools"):
            mcp_agents[name] = module.get_tools()
    return mcp_agents


# -------------------------------
# Parent reasoning agent
# -------------------------------
def create_parent_agent(verbose=True):
    """
    Creates parent agent with reasoning ability.
    This agent decides which MCP agent to use.
    """
    mcp_agents = load_mcp_agents()
    # Flatten all tools from all MCPs
    tools = [tool for group in mcp_agents.values() for tool in group]

    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent="zero-shot-react-description",
        verbose=verbose,
    )
    return agent
```

---

## 🌤️ `mcp_agents/weather_agent.py`

```python
# mcp_agents/weather_agent.py

from langchain.agents import Tool
import random

def get_weather(location: str) -> str:
    """Mock weather report (replace with API later)."""
    temps = [28, 30, 32, 35, 29]
    return f"The weather in {location} is {random.choice(temps)}°C and sunny."

def get_tools():
    return [
        Tool(
            name="Weather",
            func=get_weather,
            description="Useful for checking weather of a given location. Input should be a city name.",
        )
    ]
```

---

## 🧮 `mcp_agents/math_agent.py`

```python
# mcp_agents/math_agent.py

from langchain.agents import Tool

def calculate(expression: str) -> str:
    """Simple safe math evaluator."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"The result of {expression} is {result}"
    except Exception:
        return "Invalid math expression."

def get_tools():
    return [
        Tool(
            name="Math",
            func=calculate,
            description="Useful for performing arithmetic or math operations.",
        )
    ]
```

---

## 🔎 `mcp_agents/search_agent.py`

```python
# mcp_agents/search_agent.py

from langchain.agents import Tool

def search(query: str) -> str:
    """Mock search results (replace with real API like Tavily or SerpAPI)."""
    return f"Fake search results for: {query}"

def get_tools():
    return [
        Tool(
            name="Search",
            func=search,
            description="Useful for looking up factual or general knowledge information.",
        )
    ]
```

---

## 🚀 `main.py`

```python
# main.py
"""
Main entry — interactive CLI for dynamic parent agent
"""

from parent_agent import create_parent_agent

def main():
    print("🤖 Dynamic Multi-Agent System (type 'exit' to quit)\n")

    agent = create_parent_agent(verbose=True)

    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("👋 Goodbye!")
            break

        try:
            response = agent.run(user_input)
            print("\nAgent Response:", response, "\n")
        except Exception as e:
            print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()
```

---

## 🧩 `.env`

```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

---

## 🧪 Example Run

```
🤖 Dynamic Multi-Agent System (type 'exit' to quit)

You: What is the capital of France?
> Agent selects: Search
Agent Response: Fake search results for: capital of France

You: What is the weather in Mumbai?
> Agent selects: Weather
Agent Response: The weather in Mumbai is 30°C and sunny.

You: What is 15 * 12?
> Agent selects: Math
Agent Response: The result of 15 * 12 is 180.

You: exit
👋 Goodbye!
```

---

## 🔮 Highlights

✅ **Dynamic Discovery** — You can add new MCP agents easily; just drop a new file in `mcp_agents/` with a `get_tools()` function.
✅ **Parent Reasoning** — The main LLM uses ReAct reasoning to decide which tool (MCP) to call.
✅ **Extensible** — Ready for LangGraph, FastAPI, or async integration.
✅ **Modular** — Each MCP agent is isolated, testable, and plug-and-play.

---

Would you like me to now convert this **multi-agent system into a LangGraph graph**, so each MCP agent becomes a **LangGraph node** with dynamic routing via edges?
(That will make it fully visualizable and scalable for workflow orchestration.)
